In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 


In [47]:

root2 = math.sqrt(2)
root3 = math.sqrt(3)

denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 

#A1 = X, diagonal basis measurement
#A3 = Z, standard basis measurement

random_1in3_transform_matrix = [ [ 1 / root3 , - root2 / root3 ],
                                  [ root2 / root3, 1 / root3 ] ]
random_1in3_transform = Operator(random_1in3_transform_matrix)

B1_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
B1_transform = Operator(B1_transform_matrix) 
A2_transform = B1_transform
#W = B1_transform


B3_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
B3_transform = Operator(B3_transform_matrix) 
#V = B3_transform


def random():
    q = QuantumCircuit(1) 
    q.h(0) 
    q.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    return counts.get("1",0)


def random_1in3():
    q = QuantumCircuit(1)
    #q = (1, 0)
    q.unitary(random_1in3_transform, [0]) #1/root3 |0> + 2/root3 |1>

    q.measure_all()
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    output = counts.get("1", 0) #1/3 chance for 0- take as output if so
    if output == 0:
        return 1 #Return as 1

    #Otherwise, as 2/3 chance for 1, do a 1/2 random to choose between 1 and 2 - giving 2/3 * 1/2 = 1/3 chance for each
    #Return as 2 and 3
    return 2 + random()

def randomList(n):
    bs = []
    for i in range(n):
        bs = bs + [random()] 
    return bs 

def randomList_1in3(n):
    bs = []
    for i in range(n):
        bs = bs + [random_1in3()]
    return bs

# Given a circuit c which prepares the state 1/sqrt(2)( |01> - |10> ) and measures it, 
# simulate it n times and calculate the average measurement result (using the conversion to +1 and -1)

def average(c,n): 
    backend = BasicSimulator()
    compiled = transpile(c, backend)
    job_sim = backend.run(compiled, shots=n)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    count00 = counts.get("00",0) 
    count01 = counts.get("01",0) 
    count10 = counts.get("10",0) 
    count11 = counts.get("11",0) 
    # The return value includes the conversion from measurement results 0,1 to +1,-1
    # Each 00 means a value of  1 (+1 * +1)
    # Each 01 means a value of -1 (+1 * -1)
    # Each 10 means a value of -1 (-1 * +1)
    # Each 11 means a value of  1 (+1 * +1)
    return (count00 - count01 - count10 + count11) / n 


def measurement_increment(c):
    backend = BasicSimulator()
    compiled = transpile(c, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    count = result_sim.get_counts(compiled)
    result = list(count.keys())[0] #one key from one shot
    if result[0] == result[1]:#00 or 11
        return 1
    else: #01 or 10
        return -1
    

# Construct a circuit that prepares the state  1/sqrt(2)( |01> - |10> )
def entangledPair():
    q = QuantumCircuit(2, 2) 
    q.h(0)
    q.cx(0,1)
    q.x(0)
    q.z(1)
    return q


#Intercept B qbit, measure on random B basis, reset and return
def attack(circuit, attacker_basis):

    if attacker_basis == 1:   # W basis
        circuit.unitary(B1_transform, [1])
        circuit.measure(1,0)

        #Reset qbit and rotate depending on measurment
        #Include inverse (unitary so just transpose) of the transformation
        circuit.reset(1)
        circuit.unitary(B1_transform.transpose(), [1])

    elif attacker_basis == 2: # Z basis
        circuit.measure(1,0)

        circuit.reset(1)

    else:                     # V basis
        circuit.unitary(B3_transform, [1])
        circuit.measure(1,0)

        circuit.reset(1)
        circuit.unitary(B3_transform.transpose(), [1])

    return circuit


def A1_B1_measurement(circuit_A1_B1, attacker):
    #A1 = X, diagonal basis measurement
    # Transform qubit 0 by H
    circuit_A1_B1.h(0)


    #if attacker, intercept Bs bit and measure in random basis and resend
    if attacker > 0:
        circuit_A1_B1 = attack(circuit_A1_B1, attacker)
    
    
    # Transform qubit 1 by B1_transform (W)
    circuit_A1_B1.unitary(B1_transform,[1])
    # Measure both qubits
    circuit_A1_B1.measure(range(2), range(2))
    return measurement_increment(circuit_A1_B1) 
    
    

def A1_B3_measurement(circuit_A1_B3, attacker):
    #A1 = X, diagonal basis measurement
    # Transform qubit 0 by H
    circuit_A1_B3.h(0)

    #if attacker, intercept Bs bit and measure in random basis and resend
    if attacker > 0:
        circuit_A1_B3 = attack(circuit_A1_B3, attacker)
    
    
    # Transform qubit 1 by B3_transform (V)
    circuit_A1_B3.unitary(B3_transform,[1])
    # Measure both qubits
    circuit_A1_B3.measure(range(2), range(2))
    return measurement_increment(circuit_A1_B3)

    

def A3_B1_measurement(circuit_A3_B1, attacker):
    # Don't transform qubit 0 - transform Z, standard basis measurement

    #if attacker, intercept Bs bit and measure in random basis and resend
    if attacker > 0:
        circuit_A3_B1 = attack(circuit_A3_B1, attacker)
    
    
    # Transform qubit 1 by B1_transform (W)
    circuit_A3_B1.unitary(B1_transform,[1])
    # Measure both qubits
    circuit_A3_B1.measure(range(2), range(2))
    return measurement_increment(circuit_A3_B1)
    
def A3_B3_measurement(circuit_A3_B3, attacker):
    # Don't transform quibit 0 - transform Z, standard basis measurement

    #if attacker, intercept Bs bit and measure in random basis and resend
    if attacker > 0:
        circuit_A3_B3 = attack(circuit_A3_B3, attacker)
    
    # Transform qubit 1 by B3_transform (V)
    circuit_A3_B3.unitary(B3_transform,[1])
    # Measure both qubits
    circuit_A3_B3.measure(range(2), range(2))
    return measurement_increment(circuit_A3_B3)

def key_bit_measurement(c):
    backend = BasicSimulator()
    compiled = transpile(c, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    #guaranteed to be opposite - 01 or 10
    if counts.get("01", 0) == 1:
        return 0, 0 #Return As bit, and Bs bit negated

    elif counts.get("10", 0) == 1:
        return 1,1 #Return As bit, and Bs bit negated

    #not oppsosite, error, attacker present? A and B cannot check at this point though
    elif counts.get("00", 0) == 1:
        return 0,1

    return 1,0
    

def A2_B1_measurement(circuit_A2_B1, attacker):
    # Transform qubit 0 by A2_transform (W)
    circuit_A2_B1.unitary(A2_transform, [0])
    # Transform qubit 1 by B1_transform (W)

    #if attacker, intercept Bs bit and measure in random basis and resend
    if attacker > 0:
        circuit_A2_B1 = attack(circuit_A2_B1, attacker)
    
    circuit_A2_B1.unitary(B1_transform,[1])
    # Measure both qubits
    circuit_A2_B1.measure(range(2), range(2))
    return key_bit_measurement(circuit_A2_B1)
    

def A3_B2_measurement(circuit_A3_B2, attacker):

    #if attacker, intercept Bs bit and measure in random basis and resend
    if attacker > 0:
        circuit_A3_B2 = attack(circuit_A3_B2, attacker)
    
    
    #Both Z, so just measure in standard basis, no transform
    circuit_A3_B2.measure(range(2), range(2))
    return key_bit_measurement(circuit_A3_B2)
    
    


In [49]:

#desired key length
key_length = 200
#process repeats
N = int((9*key_length)/2)
#Get int x in range 1 to 8 to get N/x attacks
n_attacks = int( str(random())+str(random())+str(random()), 2) + 1
print("process repeats: ", N)
print(n_attacks)
print("number of attacks", int(N/n_attacks))

#choose random transforms to measure
A_transforms = randomList_1in3(N)
B_transforms = randomList_1in3(N)
attacker_transforms = randomList_1in3(int(N/n_attacks)) 
A_key_measurements = []
B_key_measurements = []

#Validation live count, and number of measurements, to divide by at end
avg_counts = {
        "11" : [0,0], "13" : [0,0], "31" : [0,0], "33" : [0,0]
        #idx 0 is live tracker of +1 -1 vals, idx 1 is n count
    }

avg_i_j_to_func = {
        "11" : A1_B1_measurement, "13" : A1_B3_measurement, "31" : A3_B1_measurement, "33" : A3_B3_measurement
    }


#measure for each transform
for idx in range(N):
    i = A_transforms[idx]
    j = B_transforms[idx]
    try: #Run n_attacks attacks
        e = attacker_transforms[idx]
    except:
        e = -1
    transform_combo = str(i) + str(j)
    circuit = entangledPair()

    if transform_combo in avg_counts.keys():
        result = avg_i_j_to_func[transform_combo](circuit, e)
        avg_counts[transform_combo][0] += result
        avg_counts[transform_combo][1] += 1

    elif transform_combo == "32":
        A_key_bit, B_key_bit = A3_B2_measurement(circuit, e)
        A_key_measurements.append(A_key_bit)
        B_key_measurements.append(B_key_bit) #Already inverted in function

    elif transform_combo == "21":
        A_key_bit, B_key_bit = A2_B1_measurement(circuit, e)
        A_key_measurements.append(A_key_bit)
        B_key_measurements.append(B_key_bit) #Already inverted in function

#Calculate averages
for transform_combo, counts in avg_counts.items():
    #Transform in same dict from count to average
    avg_counts[transform_combo] = avg_counts[transform_combo][0]/avg_counts[transform_combo][1]
#Calculate S
X_W = avg_counts["11"]
X_V = avg_counts["13"]
Z_W = avg_counts["31"]
Z_V = avg_counts["33"]

S = abs(X_W - X_V + Z_W + Z_V)
print(S)

#Found in testing S can dip quite low even with no attacks. Check for below 2.65 to raise suspicion, as with low number of attacks 
#it can still get very close to 2root2. 
#but there is probably a better checking method
if S < 2.25:
    print("Definite inteference detected. Attacker very likely")
elif S < 2.65:
    print("Unusual result detected. qBits likley not entangled. Potential Attacker")
else:
    print("qbits entangled")

print(A_key_measurements == B_key_measurements)


process repeats:  900
6
number of attacks 150
2.4192957312450454
Unusual result detected. qBits likley not entangled. Potential Attacker
False
